# Crime Risk Assessment using Spatial Analysis and Tree-Based Learning

End-to-end pipeline on the **Chicago Crimes (2001-present)** dataset (~7.8M records):
grid-based hotspot detection, risk labelling, privacy controls, XGBoost and Random Forest models, SHAP explainability and case-similarity retrieval.

**Contents**
0. Setup
1. Load Dataset
2. Data Preprocessing & Cleaning
3. Exploratory Data Analysis
4. Grid-Based Spatial Aggregation & Risk Labelling
5. Data Security & Privacy Controls
6. Crime Risk Prediction (XGBoost vs Random Forest)
7. Explainability (SHAP)
8. Crime Category Prediction
9. Case Similarity System
10. End-to-End Inference on a New Case

## 0. Setup

In [ ]:
# If running on Colab, uncomment:
# !pip install shap xgboost

In [ ]:
import gc
import json
import random
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc
)
from sklearn.metrics.pairwise import cosine_similarity

from xgboost import XGBClassifier, plot_importance
import shap

## 1. Load Dataset

Download instructions: see `data/README.md`.

In [ ]:
DATA_PATH = "../data/Crimes_-_2001_to_Present.csv"

# On Google Colab (dataset stored in Drive):
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/Crimes_-_2001_to_Present.csv"

In [ ]:
use_cols = [
    'Date','Primary Type','Latitude','Longitude',
    'Arrest','Domestic','District','Community Area',
    'Location Description',"ID", "Case Number","Ward"
]

df = pd.read_csv(DATA_PATH, usecols=use_cols, low_memory=True)
print("Dataset Shape",df.shape)
print("\nColumns",df.columns)
print("\nData Types",df.dtypes)
print("\nMissing Values",df.isnull().sum())

### Class information

In [ ]:
# Number of Unique Crime Classes
num_classes = df["Primary Type"].nunique()
print("\nNumber of Crime Classes:", num_classes)

# List All Unique Crime Types
print("\nAll Crime Classes:")
print(df["Primary Type"].unique())

# Count of Each Crime Type
crime_counts = df["Primary Type"].value_counts()
print("\nCrime Counts:")
print(crime_counts)

# Percentage Distribution
crime_percent = df["Primary Type"].value_counts(normalize=True) * 100
print("\nCrime Percentage Distribution:")
print(crime_percent)

### Crime category grouping

In [ ]:
crime_mapping = {
'THEFT':'Property Crime',
'BURGLARY':'Property Crime',
'MOTOR VEHICLE THEFT':'Property Crime',
'CRIMINAL DAMAGE':'Property Crime',

'BATTERY':'Violent Crime',
'ASSAULT':'Violent Crime',
'HOMICIDE':'Violent Crime',

'NARCOTICS':'Drug Crime',
'OTHER NARCOTIC VIOLATION':'Drug Crime',

# Public Disorder
'PUBLIC PEACE VIOLATION':'Other',
'GAMBLING':'Other',
'LIQUOR LAW VIOLATION':'Other',
'INTERFERENCE WITH PUBLIC OFFICER':'Other',

# Financial Crimes
'DECEPTIVE PRACTICE':'Other',
'COUNTERFEITING':'Other',
'EMBEZZLEMENT':'Other',
'FORGERY':'Other',

# Safety / Regulatory
'WEAPONS VIOLATION':'Other',
'SEX OFFENSE':'Other',
'OFFENSE INVOLVING CHILDREN':'Other',
'PROSTITUTION':'Other',
'STALKING':'Other'
}

df['Crime_Category'] = df['Primary Type'].map(crime_mapping).fillna('Other')
print(df['Crime_Category'].value_counts())

## 2. Data Preprocessing & Cleaning

### Removing Duplicate cases

In [ ]:
df = df.drop_duplicates(subset=["ID", "Case Number"])
print("After removing duplicates:", df.shape)

### Removing noisy/irrelevant columns

In [ ]:
cols_to_drop = [
    "ID", "Case Number"
]

df = df.drop(columns=cols_to_drop)
print("After dropping columns:", df.shape)

### Convert date to datetime format

In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

### Create temporal features

In [ ]:
df['Year'] = df['Date'].dt.year.astype('int16')
df['Month'] = df['Date'].dt.month.astype('int8')
df['Hour'] = df['Date'].dt.hour.astype('int8')
df['DayOfWeek'] = df['Date'].dt.dayofweek.astype('int8')

df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype('int8')

df.drop(columns=['Date'], inplace=True)

### Boolean to numeric

In [ ]:
df["Arrest"] = df["Arrest"].astype('int8')
df["Domestic"] = df["Domestic"].astype('int8')

### Latitude and longitude

In [ ]:
df = df.dropna(subset=["Latitude", "Longitude"])

print("Rows used for Spatial Analysis:", df.shape)
print("\nColumns",df.columns)

In [ ]:
coords = df[["Longitude", "Latitude"]].values

### Handle missing data

In [ ]:
df["Location Description"] = df["Location Description"].fillna("Unknown").astype('category')

In [ ]:
df["Ward"] = df["Ward"].fillna(-1).astype('int16')
df["Community Area"] = df["Community Area"].fillna(-1).astype('int16')

In [ ]:
df = df.dropna(subset=["District"])

### Categorical feature encoding

In [ ]:
df["Primary Type Name"] = df["Primary Type"]

In [ ]:
cat_cols = [
    "Primary Type",
    "Location Description",
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col]).astype('int16')

### Sanity filtering

In [ ]:
df = df.loc[df["Year"] >= 2001]

## 3. Exploratory Data Analysis

In [ ]:
df["Year"].value_counts().sort_index().plot(
    kind="line", figsize=(8,4), title="Crime Count by Year"
)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

df["Primary Type Name"].value_counts().head(10).plot(
    kind="bar"
)

plt.title("Top 10 Crime Types")
plt.xlabel("Crime Type")
plt.ylabel("Number of Crimes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(
    df["Longitude"],
    df["Latitude"],
    s=1,
    alpha=0.3
)

plt.title("Crime Locations in Chicago (Valid Coordinates)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.xlim(-87.95, -87.50)
plt.ylim(41.60, 42.10)
plt.show()

## 4. Grid-Based Spatial Aggregation & Risk Labelling

KDE was computationally infeasible at this scale, so crimes are aggregated into ~1 km grid cells.

In [ ]:
CHI_LAT_MIN, CHI_LAT_MAX = 41.6, 42.05
CHI_LON_MIN, CHI_LON_MAX = -87.95, -87.5

df = df.loc[
    (df["Latitude"].between(CHI_LAT_MIN, CHI_LAT_MAX)) &
    (df["Longitude"].between(CHI_LON_MIN, CHI_LON_MAX))
]

### Create Spatial Grid

In [ ]:
grid_size = 0.01  # ~1 km grid

df["lat_bin"] = (df["Latitude"] / grid_size).astype('int16')
df["lon_bin"] = (df["Longitude"] / grid_size).astype('int16')

### Crimes Per grid cell

In [ ]:
grid_counts = (
    df[["lat_bin", "lon_bin"]]
    .value_counts()
    .reset_index(name="Crime_Count")
)

### Risk labelling (all years)

Used for hotspot mapping and case-similarity metadata only. The prediction model in Section 6 uses period-specific labels to avoid leakage.

In [ ]:
low_thr  = grid_counts["Crime_Count"].quantile(0.33)
high_thr = grid_counts["Crime_Count"].quantile(0.66)
print("lower threshold",low_thr)
print("higher threshold",high_thr)
def risk_label(c):
    if c >= high_thr:
        return "High"
    elif c >= low_thr:
        return "Medium"
    else:
        return "Low"

grid_counts["Crime_Risk"] = grid_counts["Crime_Count"].apply(risk_label)
grid_counts["Crime_Risk"] = grid_counts["Crime_Risk"].astype('category')
grid_counts["Crime_Count"] = grid_counts["Crime_Count"].astype('int32')

### Chicago Crime hotspots plot

In [ ]:
sample_size = min(50000, len(grid_counts))
grid_sample = grid_counts.sample(sample_size, random_state=42)
# Convert bins back to real coordinates
lat_plot = grid_sample["lat_bin"] * grid_size
lon_plot = grid_sample["lon_bin"] * grid_size

plt.figure(figsize=(8,8))
plt.scatter(
    lon_plot,
    lat_plot,
    c=grid_sample["Crime_Count"],
    cmap="hot",
    s=10
)

plt.colorbar(label="Crime Count")
plt.title("Chicago Neighborhood Crime Hotspots (Grid-based)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.xlim(CHI_LON_MIN, CHI_LON_MAX)
plt.ylim(CHI_LAT_MIN, CHI_LAT_MAX)

plt.gca().set_aspect('equal', adjustable='box')
plt.show()

### Normalize grid crime counts to a Hotspot Score

In [ ]:
# min max normalisation
grid_counts["Hotspot_Score"] = (
    grid_counts["Crime_Count"] - grid_counts["Crime_Count"].min()
) / (
    grid_counts["Crime_Count"].max() - grid_counts["Crime_Count"].min()
)
grid_counts["Hotspot_Score"] = grid_counts["Hotspot_Score"].astype('float32')

In [ ]:
grid_counts["neighbor_density"] = (
    grid_counts["Crime_Count"]
    .rolling(window=3, center=True)
    .mean()
    .fillna(0)
    .astype('float32')
)

### Attach hotspot score to EACH crime record

In [ ]:
# Create same grid bins for original data
grid_size = 0.01  # same as before

# Merge hotspot score
df = df.merge(
    grid_counts[["lat_bin", "lon_bin", "Hotspot_Score", "Crime_Risk"]],
    on=["lat_bin", "lon_bin"],
    how="left",
    sort=False
)

### Handle missing hotspot values (very rare)

In [ ]:
df["Hotspot_Score"] = df["Hotspot_Score"].fillna(0).astype('float32')
df["Crime_Risk"] = df["Crime_Risk"].fillna("Low").astype('category')

In [ ]:
df["Hotspot_Score"].sample(200000).hist(bins=50)

In [ ]:
df[
    ["Latitude", "Longitude", "Hotspot_Score", "Crime_Risk"]
].head(50)

In [ ]:
print(df["Hotspot_Score"].describe())
print((df["Hotspot_Score"] == 0).sum())

## 5. Data Security & Privacy Controls

Privacy-preserving techniques and access controls applied to the processed dataset before model training.

In [ ]:
# PII & Sensitive Column Audit
sensitive_cols = ['Latitude', 'Longitude', 'District', 'Ward',
                  'Community Area', 'Beat', 'Location Description']

present_sensitive = [c for c in sensitive_cols if c in df.columns]
print("Sensitive columns still present in dataset:")
for col in present_sensitive:
    print(f"  - {col}")

In [ ]:
# Coordinate Masking (Spatial Perturbation)
# Round to 3 decimal places ≈ ~100m precision instead of exact GPS
PRECISION = 3

df_secure = df.copy()

if 'Latitude' in df_secure.columns:
    df_secure['Latitude'] = df_secure['Latitude'].round(PRECISION)
if 'Longitude' in df_secure.columns:
    df_secure['Longitude'] = df_secure['Longitude'].round(PRECISION)

print("Coordinates masked to", PRECISION, "decimal place precision.")
print(df_secure[['Latitude', 'Longitude']].head())

In [ ]:
# Suppress Rare Categories (k-Anonymity)
K_THRESHOLD = 5

# Apply to Primary Type Name (original crime labels before encoding)
if 'Primary Type Name' in df_secure.columns:
    type_counts = df_secure['Primary Type Name'].value_counts()
    rare_types = type_counts[type_counts < K_THRESHOLD].index
    df_secure['Primary Type Name'] = df_secure['Primary Type Name'].apply(
        lambda x: 'Other' if x in rare_types else x
    )
    print(f"Suppressed {len(rare_types)} rare crime type(s) with fewer than {K_THRESHOLD} occurrences.")

In [ ]:
# Role-Based Access Control (Column Views)
# Public/Reporting view - no precise location data
public_view_cols = [
    'Primary Type Name', 'Crime_Risk', 'Hotspot_Score',
    'Hour', 'DayOfWeek', 'Month', 'IsWeekend',
    'Arrest', 'Domestic', 'Year'
]

# Model training view - includes spatial bins but NOT raw GPS
model_view_cols = [
    'Primary Type','Crime_Category','Description', 'Location Description',
    'IUCR', 'FBI Code', 'Beat', 'District', 'Ward', 'Community Area',
    'Arrest', 'Domestic', 'Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'Year',
    'lat_bin', 'lon_bin', 'Hotspot_Score', 'Crime_Risk','neighbor_density'

]

# Filter only columns that exist
public_view = df_secure[[c for c in public_view_cols if c in df_secure.columns]]
model_view  = df_secure[[c for c in model_view_cols  if c in df_secure.columns]]

print("Public View Shape  :", public_view.shape)
print("Model Training View Shape:", model_view.shape)

In [ ]:
# Data Access Audit Log
audit_log = {
    "timestamp": datetime.now().isoformat(),
    "dataset": "Chicago Crimes 2001-Present",
    "source": "https://www.kaggle.com/datasets/adelanseur/crimes-2001-to-present-chicago",
    "records_used": int(df_secure.shape[0]),
    "columns_in_model_view": list(model_view.columns),
    "columns_in_public_view": list(public_view.columns),
    "privacy_measures_applied": [
        "Coordinate masking (3 decimal precision)",
        "k-Anonymity suppression (k=5) on rare crime types",
        "Role-based column access control",
        "Sensitive column audit"
    ]
}

# Save audit log
with open("data_access_audit_log.json", "w") as f:
    json.dump(audit_log, f, indent=4)

print("Audit log saved. Summary:")
print(json.dumps(audit_log, indent=4))

## 6. Crime Risk Prediction (XGBoost vs Random Forest)

In [ ]:
model_view.head()

In [ ]:
model_view.info()
model_view.describe()
model_view['Crime_Risk'].value_counts()

### Temporal split (prevents label leakage)

The risk labels from Section 4 are computed from **all years** and are a direct function of the grid cell (`lat_bin`, `lon_bin`), which are also model features. A random train/test split would therefore only test memorisation.

Instead, labels are rebuilt **per period**: the model learns from 2001-2019 labels and is evaluated on labels computed only from 2020+ data, answering a real question: does a cell's historical risk predict its future risk?

In [ ]:
SPLIT_YEAR = 2020   # train: 2001-2019 | test: 2020-present

def period_risk_labels(period_df):
    """Label each grid cell High/Medium/Low using ONLY this period's crime counts."""
    counts = (period_df[["lat_bin", "lon_bin"]]
              .value_counts()
              .reset_index(name="Period_Count"))
    low_thr  = counts["Period_Count"].quantile(0.33)
    high_thr = counts["Period_Count"].quantile(0.66)
    counts["Risk_Label"] = np.select(
        [counts["Period_Count"] >= high_thr, counts["Period_Count"] >= low_thr],
        ["High", "Medium"], default="Low")
    return counts[["lat_bin", "lon_bin", "Risk_Label"]]

train_period = model_view[model_view["Year"] <  SPLIT_YEAR]
test_period  = model_view[model_view["Year"] >= SPLIT_YEAR]

train_cell_labels = period_risk_labels(train_period)
test_cell_labels  = period_risk_labels(test_period)

print("Train period rows:", len(train_period), "| Test period rows:", len(test_period))

### Random Sampling

In [ ]:
train_sample = (train_period.sample(n=min(200_000, len(train_period)), random_state=42)
                .merge(train_cell_labels, on=["lat_bin", "lon_bin"], how="left"))
test_sample  = (test_period.sample(n=min(50_000, len(test_period)), random_state=42)
                .merge(test_cell_labels, on=["lat_bin", "lon_bin"], how="left"))

### Balance the TRAINING set only

Upsampling happens after the split, so duplicated rows can never appear in the test set.

In [ ]:
print("Before Balancing (train):")
print(train_sample["Risk_Label"].value_counts())

# Split classes
df_high   = train_sample[train_sample["Risk_Label"] == "High"]
df_medium = train_sample[train_sample["Risk_Label"] == "Medium"]
df_low    = train_sample[train_sample["Risk_Label"] == "Low"]

# Find largest class size
target_size = max(len(df_high), len(df_medium), len(df_low))

# Upsample smaller classes
df_high_up   = resample(df_high,   replace=True, n_samples=target_size, random_state=42)
df_medium_up = resample(df_medium, replace=True, n_samples=target_size, random_state=42)
df_low_up    = resample(df_low,    replace=True, n_samples=target_size, random_state=42)

# Combine + shuffle
train_sample = pd.concat([df_high_up, df_medium_up, df_low_up])
train_sample = train_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter Balancing (train):")
print(train_sample["Risk_Label"].value_counts())
print("\nTest set (untouched, natural distribution):")
print(test_sample["Risk_Label"].value_counts())

### Features

`Year` is excluded: train and test cover different years, so a tree model cannot generalise on it.

In [ ]:
features_risk = [
    'lat_bin',
    'lon_bin',
    'Hour',
    'DayOfWeek',
    'IsWeekend',
    'Month',
    'District'
]

X_train_r = train_sample[features_risk].copy()
X_test_r  = test_sample[features_risk].copy()

### Reduce Memory usage

In [ ]:
for X in (X_train_r, X_test_r):
    X['lat_bin']   = X['lat_bin'].astype('float32')
    X['lon_bin']   = X['lon_bin'].astype('float32')
    X['Hour']      = X['Hour'].astype('int8')
    X['DayOfWeek'] = X['DayOfWeek'].astype('int8')
    X['IsWeekend'] = X['IsWeekend'].astype('int8')
    X['Month']     = X['Month'].astype('int8')

### Encode Target Variable

In [ ]:
le_risk = LabelEncoder()
y_train_r = le_risk.fit_transform(train_sample['Risk_Label'])
y_test_r  = le_risk.transform(test_sample['Risk_Label'])

### Train XGBoost Model 1

In [ ]:
xgb_risk = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=150,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_risk.fit(X_train_r, y_train_r)

y_pred_risk_xgb = xgb_risk.predict(X_test_r)

In [ ]:
y_prob_risk_xgb = xgb_risk.predict_proba(X_test_r)

### Feature Importance

In [ ]:
plot_importance(xgb_risk)
plt.title("Feature Importance")
plt.show()

### Train Random Forest Model 1

In [ ]:
rf_risk = RandomForestClassifier(
    n_estimators=120,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_risk.fit(X_train_r, y_train_r)

y_pred_risk_rf = rf_risk.predict(X_test_r)

In [ ]:
y_prob_risk_rf = rf_risk.predict_proba(X_test_r)

### Evaluate Model

In [ ]:
print("\nXGBoost Accuracy:", accuracy_score(y_test_r, y_pred_risk_xgb))
print(classification_report(y_test_r, y_pred_risk_xgb, target_names=le_risk.classes_))

print("\nRandom Forest Accuracy:", accuracy_score(y_test_r, y_pred_risk_rf))
print(classification_report(y_test_r, y_pred_risk_rf, target_names=le_risk.classes_))

### Persistence Baseline

The simplest possible forecast: *a cell's future risk = its past risk*. The models should be judged against this.

In [ ]:
baseline_pred = (test_sample[["lat_bin", "lon_bin"]]
                 .merge(train_cell_labels, on=["lat_bin", "lon_bin"], how="left")["Risk_Label"]
                 .fillna("Low"))   # cells with no pre-2020 crimes
y_pred_baseline = le_risk.transform(baseline_pred)

print("Persistence Baseline Accuracy:", accuracy_score(y_test_r, y_pred_baseline))
print(classification_report(y_test_r, y_pred_baseline, target_names=le_risk.classes_))

### Feature Importance

In [ ]:
pd.Series(rf_risk.feature_importances_, index=features_risk).plot(kind='bar')
plt.show()

### Confusion Matrix

In [ ]:
# XGBoost
cm = confusion_matrix(y_test_r, y_pred_risk_xgb)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("XGB Risk Confusion Matrix")
plt.show()

# Random Forest
cm = confusion_matrix(y_test_r, y_pred_risk_rf)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("RF Risk Confusion Matrix")
plt.show()

### Prediction Probability

In [ ]:
print("XGB Risk Probabilities:\n", y_prob_risk_xgb[:5])
print("RF Risk Probabilities:\n", y_prob_risk_rf[:5])

### ROC/AUC Curve

In [ ]:
y_test_bin = label_binarize(y_test_r, classes=[0,1,2])

# XGBoost
for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_risk_xgb[:, i])
    plt.plot(fpr, tpr)
plt.title("ROC - XGB Risk")
plt.show()

# Random Forest
for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_risk_rf[:, i])
    plt.plot(fpr, tpr)
plt.title("ROC - RF Risk")
plt.show()

## 7. Explainability (SHAP)

Explains individual predictions of the XGBoost risk model and maps key drivers to actionable suggestions.

In [ ]:
explainer = shap.Explainer(xgb_risk)

In [ ]:
shap_values = explainer.shap_values(X_test_r)

In [ ]:
shap.summary_plot(shap_values, X_test_r)

In [ ]:
sample_index = random.randint(0, len(X_test_r)-1)

sample_index = 0
class_index = 0

In [ ]:
pred = xgb_risk.predict(X_test_r.iloc[[sample_index]])[0]
class_index = int(pred)
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[sample_index, :, class_index],
        base_values=explainer.expected_value[class_index],
        data=X_test_r.iloc[sample_index]
    )
)

In [ ]:
print("Predicted Risk Class:", pred)

In [ ]:
print("Predicted Risk:", le_risk.inverse_transform([pred])[0])

In [ ]:
shap_row = shap_values[sample_index, :, class_index]
sample = X_test_r.iloc[sample_index]

In [ ]:
def generate_explanation(shap_row, sample):
    explanation = []

    for feature, value in zip(sample.index, shap_row):
        if value > 0:
            explanation.append(f"{feature} increases risk")
        else:
            explanation.append(f"{feature} decreases risk")

    return explanation

print("\nReasons:")
print(generate_explanation(shap_row, sample))

In [ ]:
def generate_suggestions(shap_row, sample):
    suggestions = []

    for feature, value in zip(sample.index, shap_row):
        if value > 0:

            if feature == 'Hotspot_Score':
                suggestions.append("Increase police patrolling in this area")

            elif feature == 'Hour':
                suggestions.append("Deploy more night patrol units")

            elif feature == 'District':
                suggestions.append("Focus surveillance in this district")

            elif feature == 'DayOfWeek':
                suggestions.append("Increase monitoring on high-risk days")

    return suggestions

print("\nSuggestions:")
print(generate_suggestions(shap_row, sample))

In [ ]:
print(" Predicted Risk:", le_risk.inverse_transform([pred])[0])

print("\n Key Factors:")
for reason in generate_explanation(shap_row, sample)[:5]:
    print("-", reason)

print("\n Recommended Actions:")
for sug in generate_suggestions(shap_row, sample):
    print("-", sug)

## 8. Crime Category Prediction

### Sample model

In [ ]:
model_sample_type = model_view.sample(n=100000, random_state=42)

### Remove other

In [ ]:
model_sample_type = model_sample_type[
    model_sample_type["Crime_Category"] != "Other"
]

### Features & Target

In [ ]:
features_type = [
    'Year',
    'lat_bin',
    'lon_bin',
    'Hour',
    'DayOfWeek',
    'IsWeekend',
    'Month',
    'District',
    'Hotspot_Score',
]

X_type = model_sample_type[features_type].copy()
y_type = model_sample_type['Crime_Category']

### Feature Encoding

In [ ]:
le_type = LabelEncoder()
y_type = le_type.fit_transform(y_type)

# Encode District (important)
le_district = LabelEncoder()
X_type['District'] = le_district.fit_transform(X_type['District'])

### Train Test Split

In [ ]:
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_type, y_type,
    test_size=0.2,
    stratify=y_type,
    random_state=42
)

### XGBoost Model 2

In [ ]:
xgb_type = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le_type.classes_),
    n_estimators=120,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_t
)

xgb_type.fit(X_train_t, y_train_t, sample_weight=sample_weights)

y_pred_type_xgb = xgb_type.predict(X_test_t)
y_prob_type_xgb = xgb_type.predict_proba(X_test_t)

### Random Forest Model 2

In [ ]:
rf_type = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rf_type.fit(X_train_t, y_train_t)

y_pred_type_rf = rf_type.predict(X_test_t)
y_prob_type_rf = rf_type.predict_proba(X_test_t)

### Evaluation Metrics

In [ ]:
print("\nXGBoost Accuracy:", accuracy_score(y_test_t, y_pred_type_xgb))
print(classification_report(y_test_t, y_pred_type_xgb, target_names=le_type.classes_))

print("\nRandom Forest Accuracy:", accuracy_score(y_test_t, y_pred_type_rf))
print(classification_report(y_test_t, y_pred_type_rf, target_names=le_type.classes_))

### Confusion Matrix

In [ ]:
# XGBoost
cm = confusion_matrix(y_test_t, y_pred_type_xgb)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("XGB Type Confusion Matrix")
plt.show()

# Random Forest
cm = confusion_matrix(y_test_t, y_pred_type_rf)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("RF Type Confusion Matrix")
plt.show()

### ROC AUC Score

In [ ]:
y_test_bin_t = label_binarize(y_test_t, classes=list(range(len(le_type.classes_))))

# XGBoost
for i in range(len(le_type.classes_)):
    fpr, tpr, _ = roc_curve(y_test_bin_t[:, i], y_prob_type_xgb[:, i])
    plt.plot(fpr, tpr)
plt.title("ROC - XGB Type")
plt.show()

# Random Forest
for i in range(len(le_type.classes_)):
    fpr, tpr, _ = roc_curve(y_test_bin_t[:, i], y_prob_type_rf[:, i])
    plt.plot(fpr, tpr)
plt.title("ROC - RF Type")
plt.show()

### Probabilities

In [ ]:
print("XGB Type Probabilities:\n", y_prob_type_xgb[:5])
print("RF Type Probabilities:\n", y_prob_type_rf[:5])

## 9. Case Similarity System

Retrieves the most similar historical cases (cosine similarity on scaled features), retrains both models and renders a summary dashboard.

In [ ]:
# FEATURE CONFIG
FEATURES_SIMILARITY = [
    'Year', 'lat_bin', 'lon_bin',
    'Hour', 'DayOfWeek', 'IsWeekend',
    'Month', 'District', 'Hotspot_Score'
]
TARGET_RISK = 'Crime_Risk'
TARGET_TYPE = 'Crime_Category'

In [ ]:
# STEP 1 - BUILD FEATURE VECTORS (same preprocessing)

def build_feature_matrix(df):
    df = df.copy()
    le_dist = LabelEncoder()
    df['District'] = le_dist.fit_transform(df['District'].astype(str))
    feat_cols = [c for c in FEATURES_SIMILARITY if c in df.columns]
    X = df[feat_cols].astype(np.float32)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X).astype(np.float32)
    return X_scaled, scaler, le_dist, feat_cols

In [ ]:
# STEP 2 - CASE SIMILARITY RETRIEVAL (cosine, batch-wise)

def retrieve_similar_cases(query_idx, X_scaled, df_meta,
                            top_n=5, batch_size=20_000):
    query_vec = X_scaled[query_idx : query_idx + 1]
    n         = len(X_scaled)
    scores    = np.empty(n, dtype=np.float32)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        scores[start:end] = cosine_similarity(
            query_vec, X_scaled[start:end])[0]
    scores[query_idx] = -1.0
    top_idx = np.argpartition(scores, -top_n)[-top_n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    result  = df_meta.iloc[top_idx].copy()
    result.insert(0, 'Similarity_Score',
                  np.round(scores[top_idx].astype(float), 4))
    result.index = range(1, top_n + 1)
    return result

In [ ]:
# STEP 3 - CRIME RISK & TYPE PREDICTION

def train_models(model_view):

    # (a) CRIME RISK MODEL - temporal split, train-only balancing
    tr_period = model_view[model_view['Year'] <  SPLIT_YEAR]
    te_period = model_view[model_view['Year'] >= SPLIT_YEAR]
    tr = (tr_period.sample(n=min(200_000, len(tr_period)), random_state=42)
          .merge(period_risk_labels(tr_period), on=['lat_bin', 'lon_bin'], how='left'))
    te = (te_period.sample(n=min(50_000, len(te_period)), random_state=42)
          .merge(period_risk_labels(te_period), on=['lat_bin', 'lon_bin'], how='left'))

    max_size = tr['Risk_Label'].value_counts().max()
    parts = [resample(tr[tr['Risk_Label'] == cls], replace=True,
                      n_samples=max_size, random_state=42)
             for cls in tr['Risk_Label'].unique()]
    tr = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

    feat_r = features_risk
    X_tr_r = tr[feat_r].astype(np.float32)
    X_te_r = te[feat_r].astype(np.float32)

    le_risk = LabelEncoder()
    y_tr_r  = le_risk.fit_transform(tr['Risk_Label'])
    y_te_r  = le_risk.transform(te['Risk_Label'])

    xgb_risk = XGBClassifier(
        objective='multi:softprob', num_class=3,
        n_estimators=150, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', random_state=42, n_jobs=-1, verbosity=0)
    xgb_risk.fit(X_tr_r, y_tr_r)

    rf_risk = RandomForestClassifier(
        n_estimators=120, max_depth=12,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=42)
    rf_risk.fit(X_tr_r, y_tr_r)

    y_pred_risk_xgb = xgb_risk.predict(X_te_r)
    y_pred_risk_rf  = rf_risk.predict(X_te_r)

    print("Crime Risk - XGBoost")
    print(f"  Accuracy: {accuracy_score(y_te_r, y_pred_risk_xgb):.4f}")
    print(classification_report(y_te_r, y_pred_risk_xgb,
                                 target_names=le_risk.classes_,
                                 zero_division=0))
    print("Crime Risk - RandomForest")
    print(f"  Accuracy: {accuracy_score(y_te_r, y_pred_risk_rf):.4f}")
    gc.collect()

    # (b) CRIME TYPE MODEL
    sample_type = model_view.sample(
        n=min(100_000, len(model_view)), random_state=42)
    sample_type = sample_type[sample_type[TARGET_TYPE] != 'Other'].copy()

    feat_t = [c for c in FEATURES_SIMILARITY if c in sample_type.columns]
    X_t    = sample_type[feat_t].copy().astype(np.float32)

    le_type = LabelEncoder()
    y_t     = le_type.fit_transform(sample_type[TARGET_TYPE])

    le_d_t = LabelEncoder()
    X_t['District'] = le_d_t.fit_transform(X_t['District'].astype(str))

    X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
        X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)

    sw = compute_sample_weight('balanced', y=y_tr_t)

    xgb_type = XGBClassifier(
        objective='multi:softprob',
        num_class=len(le_type.classes_),
        n_estimators=120, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', random_state=42, n_jobs=-1, verbosity=0)
    xgb_type.fit(X_tr_t, y_tr_t, sample_weight=sw)

    rf_type = RandomForestClassifier(
        n_estimators=100, max_depth=10,
        min_samples_split=5, min_samples_leaf=2,
        max_features='sqrt', n_jobs=-1, random_state=42)
    rf_type.fit(X_tr_t, y_tr_t)

    y_pred_type_xgb = xgb_type.predict(X_te_t)
    y_prob_type_xgb = xgb_type.predict_proba(X_te_t)
    y_pred_type_rf  = rf_type.predict(X_te_t)

    print("\nCrime Type - XGBoost")
    print(f"  Accuracy: {accuracy_score(y_te_t, y_pred_type_xgb):.4f}")
    print(classification_report(y_te_t, y_pred_type_xgb,
                                 target_names=le_type.classes_,
                                 zero_division=0))
    gc.collect()

    return dict(
        xgb_risk=xgb_risk, rf_risk=rf_risk,
        le_risk=le_risk, feat_r=feat_r,
        X_te_r=X_te_r, y_te_r=y_te_r,
        y_pred_risk_xgb=y_pred_risk_xgb,
        y_pred_risk_rf=y_pred_risk_rf,
        xgb_type=xgb_type, rf_type=rf_type,
        le_type=le_type, feat_t=feat_t,
        X_te_t=X_te_t, y_te_t=y_te_t,
        y_pred_type_xgb=y_pred_type_xgb,
        y_prob_type_xgb=y_prob_type_xgb,
    )

In [ ]:
# STEP 4 - PREDICT TOP CRIMES FOR A NEW CASE

def predict_top_crimes(new_case, models, top_k=3):
    feat_t = models['feat_t']
    X_new  = new_case.copy()
    for c in feat_t:
        if c not in X_new.columns:
            X_new[c] = 0.0
    X_new   = X_new[feat_t].astype(np.float32)
    proba   = models['xgb_type'].predict_proba(X_new)[0]
    top_idx = np.argsort(proba)[::-1][:top_k]
    return pd.DataFrame({
        'Rank'       : range(1, top_k + 1),
        'Crime_Type' : models['le_type'].classes_[top_idx],
        'Probability': np.round(proba[top_idx], 4)
    })

In [ ]:
# STEP 5 - FINAL OUTPUT DASHBOARD

def plot_case_similarity_dashboard(models, similar_cases,
                                    top_crimes, new_case_label="Query Case"):
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle("Case Similarity System - Dashboard",
                 fontsize=18, fontweight='bold', y=1.01)

    # Use matplotlib.gridspec directly - avoids all namespace issues
    GS = matplotlib.gridspec.GridSpec(3, 3, figure=fig,
                                      hspace=0.55, wspace=0.40)

    COLORS = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800']

    # Panel 1: Top-N Similar Cases
    ax1    = fig.add_subplot(GS[0, :2])
    scores = similar_cases['Similarity_Score'].values
    labels = [f"Case #{i}" for i in similar_cases.index]
    if 'Crime_Category' in similar_cases.columns:
        labels = [
            f"Case #{i}  ({similar_cases.loc[i, 'Crime_Category']})"
            for i in similar_cases.index
        ]
    bars = ax1.barh(labels[::-1], scores[::-1],
                    color=COLORS[:len(scores)], edgecolor='white')
    for bar, sc in zip(bars, scores[::-1]):
        ax1.text(bar.get_width() - 0.005,
                 bar.get_y() + bar.get_height() / 2,
                 f'{sc:.4f}', va='center', ha='right',
                 color='white', fontsize=9, fontweight='bold')
    ax1.set_xlabel('Cosine Similarity Score', fontsize=10)
    ax1.set_title(
        f'Top-{len(scores)} Similar Historical Cases  |  Query: {new_case_label}',
        fontsize=11, fontweight='bold')
    ax1.set_xlim(0, 1.08)
    ax1.axvline(0.5, color='red', linestyle='--',
                linewidth=1, alpha=0.6, label='Threshold 0.5')
    ax1.legend(fontsize=8)

    # Panel 2: XGBoost Confusion Matrix
    ax2    = fig.add_subplot(GS[0, 2])
    cm_xgb = confusion_matrix(models['y_te_r'], models['y_pred_risk_xgb'])
    ConfusionMatrixDisplay(
        cm_xgb, display_labels=models['le_risk'].classes_
    ).plot(ax=ax2, colorbar=False, cmap='Blues')
    ax2.set_title('Crime Risk - XGBoost\nConfusion Matrix',
                  fontsize=10, fontweight='bold')
    ax2.tick_params(axis='x', rotation=15)

    # Panel 3: RF Confusion Matrix
    ax3   = fig.add_subplot(GS[1, 0])
    cm_rf = confusion_matrix(models['y_te_r'], models['y_pred_risk_rf'])
    ConfusionMatrixDisplay(
        cm_rf, display_labels=models['le_risk'].classes_
    ).plot(ax=ax3, colorbar=False, cmap='Greens')
    ax3.set_title('Crime Risk - RandomForest\nConfusion Matrix',
                  fontsize=10, fontweight='bold')
    ax3.tick_params(axis='x', rotation=15)

    # Panel 4: Feature Importance
    ax4         = fig.add_subplot(GS[1, 1])
    importances = models['rf_risk'].feature_importances_
    feat_labels = models['feat_r']
    sorted_idx  = np.argsort(importances)
    ax4.barh([feat_labels[i] for i in sorted_idx],
             importances[sorted_idx],
             color='#1976D2', edgecolor='white')
    ax4.set_title('Feature Importance\n(RF - Crime Risk)',
                  fontsize=10, fontweight='bold')
    ax4.set_xlabel('Importance', fontsize=9)

    # Panel 5: Predicted Crime Types (Pie)
    ax5          = fig.add_subplot(GS[1, 2])
    crime_labels = top_crimes['Crime_Type'].tolist()
    probs        = top_crimes['Probability'].tolist()
    wedge_colors = ['#EF5350', '#FFA726', '#66BB6A'][:len(probs)]
    _, _, autotexts = ax5.pie(
        probs, labels=crime_labels, colors=wedge_colors,
        autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9})
    for at in autotexts:
        at.set_fontweight('bold')
    ax5.set_title(f'Predicted Crime Types\nfor {new_case_label}',
                  fontsize=10, fontweight='bold')

    # Panel 6: Similarity Score Distribution
    ax6        = fig.add_subplot(GS[2, :])
    all_scores = similar_cases['Similarity_Score'].values
    ranks      = [f"Rank {i}" for i in similar_cases.index]
    bar_colors = plt.cm.RdYlGn(all_scores)
    b          = ax6.bar(ranks, all_scores,
                         color=bar_colors, edgecolor='white', linewidth=0.8)
    for bar, sc in zip(b, all_scores):
        ax6.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.005,
                 f'{sc:.4f}', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')
    if 'Crime_Category' in similar_cases.columns:
        for bar, (_, row) in zip(b, similar_cases.iterrows()):
            ax6.text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() / 2,
                     row['Crime_Category'],
                     ha='center', va='center',
                     fontsize=8, color='white', fontweight='bold')
    ax6.set_ylim(0, 1.15)
    ax6.set_ylabel('Cosine Similarity Score', fontsize=10)
    ax6.set_title(
        'Similarity Score per Retrieved Case  '
        '(Green = High Match  |  Red = Lower Match)',
        fontsize=11, fontweight='bold')
    ax6.axhline(0.5, color='red', linestyle='--',
                linewidth=1, alpha=0.5, label='0.5 threshold')
    ax6.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('case_similarity_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Dashboard saved as  case_similarity_dashboard.png")

In [ ]:
def run_case_similarity_system(model_view, query_row_index=0, top_n=5):
    print("=" * 60)
    print("  CASE SIMILARITY SYSTEM")
    print("=" * 60)

    # Step 1: Feature matrix
    print("\n[1/4] Building feature matrix ...")
    sample_df = (model_view
                 .sample(n=min(500_000, len(model_view)), random_state=42)
                 .reset_index(drop=True))
    X_scaled, scaler, le_dist, feat_cols = build_feature_matrix(sample_df)

    # Step 2: Similar case retrieval
    print(f"[2/4] Retrieving top-{top_n} similar cases ...")
    keep_cols = [c for c in
                 ['Crime_Category', 'Crime_Risk', 'Hour', 'DayOfWeek',
                  'Month', 'Year', 'Arrest', 'Domestic',
                  'Hotspot_Score', 'District']
                 if c in model_view.columns]
    df_meta = (model_view
               .sample(n=min(500_000, len(model_view)), random_state=42)
               .reset_index(drop=True))[keep_cols]

    similar_cases = retrieve_similar_cases(
        query_idx=query_row_index,
        X_scaled=X_scaled,
        df_meta=df_meta,
        top_n=top_n)
    print(similar_cases.to_string())
    del X_scaled
    gc.collect()

    # Step 3: Train models
    print("\n[3/4] Training prediction models ...")
    models = train_models(model_view)

    # Step 4: Predict crime type
    print("\n[4/4] Predicting crime types for query case ...")
    query_row  = model_view.iloc[[query_row_index]].copy()
    top_crimes = predict_top_crimes(query_row, models, top_k=3)
    print("\nTop Predicted Crime Types:")
    print(top_crimes.to_string(index=False))

    # Step 5: Dashboard
    print("\nGenerating dashboard ...")
    new_case_label = str(
        model_view.iloc[query_row_index]['Crime_Category']
        if 'Crime_Category' in model_view.columns
        else f"Index {query_row_index}"
    )
    plot_case_similarity_dashboard(
        models=models,
        similar_cases=similar_cases,
        top_crimes=top_crimes,
        new_case_label=new_case_label)

    return models, similar_cases, top_crimes

In [ ]:
models, similar_cases, top_crimes = run_case_similarity_system(
    model_view      = model_view,
    query_row_index = 0,
    top_n           = 5
)

## 10. End-to-End Inference on a New Case

Takes a new case, finds similar historical cases, predicts its risk, explains the prediction with SHAP and suggests actions.

In [ ]:
# STEP 1 - Build similarity matrix (only once)
X_sim_scaled, scaler_sim, le_sim_dist, sim_features = build_feature_matrix(model_view)

In [ ]:
# STEP 2 - Input (new crime case)
new_crime = {
    'Year': 2023,
    'lat_bin': int(41.88 / 0.01),
    'lon_bin': int(-87.63 / 0.01),
    'Hour': 23,
    'DayOfWeek': 5,
    'IsWeekend': 1,
    'Month': 8,
    'District': 11,
    'Hotspot_Score': 0.7
}

input_df = pd.DataFrame([new_crime])

In [ ]:
# STEP 3 - CASE SIMILARITY
temp_df = pd.concat([model_view[sim_features], input_df], ignore_index=True)
X_temp_scaled, _, _, _ = build_feature_matrix(temp_df)

query_idx = len(temp_df) - 1

similar_cases = retrieve_similar_cases(
    query_idx,
    X_temp_scaled,
    model_view,
    top_n=3
)

print("\n Similar Cases:")
print(similar_cases[['Crime_Risk','Hour','District']])

In [ ]:
# STEP 4 - PREDICTION
input_pred = input_df[features_risk].copy()

# District is kept raw, matching how xgb_risk was trained

pred = xgb_risk.predict(input_pred)[0]
pred_label = le_risk.inverse_transform([pred])[0]

print("\n Predicted Risk:", pred_label)

In [ ]:
# STEP 5 - SHAP EXPLANATION

explainer = shap.TreeExplainer(xgb_risk)
shap_values_input = explainer.shap_values(input_pred)

class_index = int(pred)
shap_row = shap_values_input[0, :, class_index]

In [ ]:
# STEP 6 - TOP FACTORS
def top_factors(shap_row, sample, top_n=5):
    pairs = list(zip(sample.columns, shap_row))
    pairs = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)

    result = []
    for f, v in pairs[:top_n]:
        if v > 0:
            result.append(f"{f} increases risk")
        else:
            result.append(f"{f} decreases risk")

    return result

factors = top_factors(shap_row, input_pred)

In [ ]:
# STEP 7 - SUGGESTIONS
def generate_suggestions(shap_row, sample):
    suggestions = []

    for feature, value in zip(sample.columns, shap_row):

        if value > 0:

            if feature in ['lat_bin','lon_bin']:
                suggestions.append("Increase police patrolling in this area")

            elif feature == 'Hour':
                suggestions.append("Deploy more night patrol units")

            elif feature == 'District':
                suggestions.append("Strengthen surveillance in this district")

            elif feature == 'DayOfWeek':
                suggestions.append("Increase monitoring on high-risk days")

    return list(set(suggestions))

suggestions = generate_suggestions(shap_row, input_pred)

In [ ]:
# STEP 8 - FINAL OUTPUT
print("\nFINAL OUTPUT ")

# print("\n Predicted Risk:", pred_label)

print("\n Key Factors:")
for f in factors:
    print("-", f)

print("\n Recommended Actions:")
for s in suggestions:
    print("-", s)